In [ ]:
# ==============================================================================
# PARALLEL DIM + FACT: STORE + SERVICE
# ==============================================================================
from notebooks.helpers import (
    IncrementalPipeline, TableConfig, get_latest_batch_id, setup_logger,
    safe_count, generate_batch_id,
)
from notebooks.helpers.silver_transforms import (
    transform_store_full_pipeline,
    transform_service_fact,
)
import pandas as pd

logger = setup_logger("parallel_dim_store_fact_service")

batch_id = generate_batch_id()
pipeline = IncrementalPipeline(spark, dbutils, batch_id=batch_id)

store_batch_id = get_latest_batch_id(spark, "store")
if not store_batch_id:
    raise ValueError("No bronze batch_id found for store; run bronze load first.")
logger.info(f"Using bronze batch_id for store: {store_batch_id}")

service_batch_id = get_latest_batch_id(spark, "service")
if not service_batch_id:
    raise ValueError("No bronze batch_id found for service; run bronze load first.")
logger.info(f"Using bronze batch_id for service: {service_batch_id}")

store_config = TableConfig(
    table_name="store",
    business_key="store_id",
    surrogate_key="store_key",
    watermark_column="last_update",
    scd_type=2,
    tracking_columns=["store_manager_id", "store_manager_first_name", "store_manager_last_name"],
    gold_table_name="dim_store",
    silver_transform=transform_store_full_pipeline,
    dependencies=["staff", "address", "city", "country"],
)

service_config = TableConfig(
    table_name="service",
    business_key="service_id",
    surrogate_key="service_key",
    watermark_column="service_date",
    scd_type=1,
    gold_table_name="fact_service",
    silver_transform=transform_service_fact,
)

print("Row Counts (Before):")
print(f"dim_store: {safe_count(spark, 'dim_store')}")
print(f"fact_service: {safe_count(spark, 'fact_service')}")
print("\nLatest Watermarks (Before):")
display(spark.table("wheelie.monitoring.watermarks"))


In [ ]:
results = []
results += pipeline.load_tables([store_config], force_full=False, bronze_batch_id=store_batch_id)
results += pipeline.load_tables([service_config], force_full=False, bronze_batch_id=service_batch_id)

display(pd.DataFrame(results))

print("\nRow Counts (After):")
print(f"dim_store: {safe_count(spark, 'dim_store')}")
print(f"fact_service: {safe_count(spark, 'fact_service')}")
print("\nLatest Watermarks (After):")
display(spark.table("wheelie.monitoring.watermarks"))
